# Hiring

## Dataset

We use the ACSEmployment dataset from the Folktables benchmark, which is derived from the U.S. Census American Community Survey (ACS). The task is a binary classification problem where the goal is to predict whether a person is employed (1) or not employed (0) based on demographic and socioeconomic features such as age, education, marital status, and work-related attributes. The dataset also includes sensitive attributes — race and sex — which are used to evaluate fairness.

### Race (RAC1P)

The variable RAC1P encodes a person’s self-identified race in the U.S. Census ACS data.

- 1: White alone
- 2: Black or African American alone
- 3: American Indian or Alaska Native alone
- 4: Alaska Native alone
- 5: American Indian alone
- 6: Asian alone
- 7: Native Hawaiian or Other Pacific Islander alone
- 8: Some other race alone
- 9: Two or more races

### Sex (SEX)

The variable SEX is binary in the ACS:

- 1: Male
- 2: Female

## Models

We train two logistic regression models on the same dataset:

### Accuracy-first model

This model uses all available features, including sensitive attributes, and is optimized purely for predictive performance (accuracy and ROC AUC). It represents a standard automated ML pipeline without fairness constraints.

### Fairness-aware model

This model excludes sensitive attributes (race and sex) from the feature set to reduce direct discrimination. While this may slightly reduce predictive accuracy, it aims to lower disparities between protected groups.

By comparing these two models, we study the trade-off between accuracy and fairness in automated machine learning pipelines.

## Data download

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from folktables import ACSDataSource, ACSEmployment

# Download and load ACS data
data_source = ACSDataSource(
    survey_year='2018',
    horizon='1-Year',
    survey='person'
)

acs_data = data_source.get_data(states=["CA"], download=True)

# Define task
task = ACSEmployment

# Features, target, sensitive attributes
X, y, sensitive = task.df_to_numpy(acs_data)

feature_names = task.features
sensitive_names = task.group

df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

# Pull sensitive attributes directly from ACS data (always present)
df["race"] = acs_data["RAC1P"].to_numpy()
df["sex"] = acs_data["SEX"].to_numpy()

## Train/Test split 

In [2]:
X = df[feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Model 1: Prioritizes accuracy. Optimizes pure performance.

In [3]:
model_acc = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model_acc.fit(X_train, y_train)

y_pred = model_acc.predict(X_test)
y_prob = model_acc.predict_proba(X_test)[:, 1]

print("Accuracy (baseline):", accuracy_score(y_test, y_pred))
print("ROC AUC (baseline):", roc_auc_score(y_test, y_prob))

Accuracy (baseline): 0.7705154602889674
ROC AUC (baseline): 0.844657697744284


## Fairness evaluation (post-hoc)

In [4]:
test_results = df.iloc[y_test.index].copy()
test_results["prob"] = y_prob

print("\nAverage predicted employment probability by race:")
print(test_results.groupby("race")["prob"].mean())

print("\nAverage predicted employment probability by sex:")
print(test_results.groupby("sex")["prob"].mean())

race_gap = (
    test_results.groupby("race")["prob"].mean().max()
    - test_results.groupby("race")["prob"].mean().min()
)

sex_gap = (
    test_results.groupby("sex")["prob"].mean().max()
    - test_results.groupby("sex")["prob"].mean().min()
)

print("\nDemographic Parity gap (race):", race_gap)
print("Demographic Parity gap (sex):", sex_gap)


Average predicted employment probability by race:
race
1    0.455207
2    0.410364
3    0.427326
4    0.356783
5    0.413929
6    0.523025
7    0.446972
8    0.437992
9    0.360427
Name: prob, dtype: float64

Average predicted employment probability by sex:
sex
1    0.492574
2    0.421806
Name: prob, dtype: float64

Demographic Parity gap (race): 0.1662416368879488
Demographic Parity gap (sex): 0.07076761542147558


# Model 2: Prioritizes fairness. Eliminates sensitive attributes

In [5]:
# Reduced feature set
fair_features = [
    f for f in feature_names
    if f not in ["RAC1P", "SEX"]
]

X_fair = df[fair_features]
y = df["target"]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fair, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_f = scaler.fit_transform(X_train_f)
X_test_f = scaler.transform(X_test_f)

## Fairness aware training

In [6]:
model_fair = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model_fair.fit(X_train_f, y_train_f)

y_pred_f = model_fair.predict(X_test_f)
y_prob_f = model_fair.predict_proba(X_test_f)[:, 1]

print("Accuracy (fairness-aware):", accuracy_score(y_test_f, y_pred_f))
print("ROC AUC (fairness-aware):", roc_auc_score(y_test_f, y_prob_f))

Accuracy (fairness-aware): 0.766766978160252
ROC AUC (fairness-aware): 0.8374121547165523


## Fairness evaluation

In [7]:
test_results_f = df.iloc[y_test_f.index].copy()
test_results_f["prob"] = y_prob_f

print("\nAverage predicted employment probability by race:")
print(test_results_f.groupby("race")["prob"].mean())

print("\nAverage predicted employment probability by sex:")
print(test_results_f.groupby("sex")["prob"].mean())

race_gap_f = (
    test_results_f.groupby("race")["prob"].mean().max()
    - test_results_f.groupby("race")["prob"].mean().min()
)

sex_gap_f = (
    test_results_f.groupby("sex")["prob"].mean().max()
    - test_results_f.groupby("sex")["prob"].mean().min()
)

print("\nDemographic Parity gap (race):", race_gap_f)
print("Demographic Parity gap (sex):", sex_gap_f)


Average predicted employment probability by race:
race
1    0.454005
2    0.409847
3    0.425227
4    0.375918
5    0.413447
6    0.526811
7    0.452419
8    0.440314
9    0.364970
Name: prob, dtype: float64

Average predicted employment probability by sex:
sex
1    0.443548
2    0.469677
Name: prob, dtype: float64

Demographic Parity gap (race): 0.16184075007551096
Demographic Parity gap (sex): 0.026129337045492973
